This pilot notebook demonstrates the feasibility of the proposed project by validating data accessibility, schema compatibility, and metric construction. It includes data loading, cleaning, county-level joins via FIPS codes, preliminary exploratory analysis, and prototype construction of the Food Infrastructure Index (FII) and Low-Access Indicator (LAI). This pilot is not intended to produce final results, but to confirm that the data pipeline and analytical framework function as expected.

In [1]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option("display.max_columns", 50)
sns.set(style="whitegrid")


In [2]:
def zscore(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    return (s - s.mean()) / s.std(ddof=0)

In [3]:
def make_fips5(x) -> str:
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = s.split(".")[0]
    return s.zfill(5)


In [4]:
def make_geoid11(x) -> str:

    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = s.split(".")[0]
    return s.zfill(11)

In [5]:
FA_PATH  = "FoodAccessResearchAtlasData2019.xlsx"
FEA_PATH = "FoodEnvironmentAtlas.xls"

DP03_PATH = "ACSDP5Y2019.DP03-Data.csv"
DP04_PATH = "ACSDP5Y2019.DP04-Data.csv"
DP05_PATH = "ACSDP5Y2019.DP05-Data.csv"

Dataset 1

In [6]:



xls = pd.ExcelFile(FA_PATH)
xls.sheet_names


['Read Me', 'Variable Lookup', 'Food Access Research Atlas']

In [7]:
food_access = pd.read_excel(
    FA_PATH,
    sheet_name="Food Access Research Atlas"
)

food_access.head()


,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,LILATracts_halfAnd10,LILATracts_1And20,LILATracts_Vehicle,HUNVFlag,LowIncomeTracts,PovertyRate,MedianFamilyIncome,LA1and10,LAhalfand10,LA1and20,LATracts_half,LATracts1,LATracts10,LATracts20,LATractsVehicle_20,...,laasian20share,lanhopi20,lanhopi20share,laaian20,laaian20share,laomultir20,laomultir20share,lahisp20,lahisp20share,lahunv20,lahunv20share,lasnap20,lasnap20share,TractLOWI,TractKids,TractSeniors,TractWhite,TractBlack,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP
0,1001020100,Alabama,Autauga County,1,1912,693,0,0.0,0.000000,0,0,0,0,0,0,11.336453,81250.0,1,1,1,1,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455.0,507.0,221.0,1622.0,217.0,14.0,0.0,14.0,45.0,44.0,6.0,102.0
1,1001020200,Alabama,Autauga County,1,2170,743,0,181.0,8.341014,1,1,1,0,0,1,17.876788,49000.0,1,1,1,1,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,802.0,606.0,214.0,888.0,1217.0,5.0,0.0,5.0,55.0,75.0,89.0,156.0
2,1001020300,Alabama,Autauga County,1,3373,1256,0,0.0,0.000000,0,0,0,0,0,0,15.046030,62609.0,1,1,1,1,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1306.0,894.0,439.0,2576.0,647.0,17.0,5.0,11.0,117.0,87.0,99.0,172.0
3,1001020400,Alabama,Autauga County,1,4386,1722,0,0.0,0.000000,0,0,0,0,0,0,2.845210,70607.0,1,1,1,1,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,922.0,1015.0,904.0,4086.0,193.0,18.0,4.0,11.0,74.0,85.0,21.0,98.0
4,1001020500,Alabama,Autauga County,1,10766,4082,0,181.0,1.681219,0,0,0,0,1,0,15.150891,96334.0,1,1,1,1,1,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2242.0,3162.0,1126.0,8666.0,1437.0,296.0,9.0,48.0,310.0,355.0,230.0,339.0


In [8]:
food_access.columns.tolist()


['CensusTract',
 'State',
 'County',
 'Urban',
 'Pop2010',
 'OHU2010',
 'GroupQuartersFlag',
 'NUMGQTRS',
 'PCTGQTRS',
 'LILATracts_1And10',
 'LILATracts_halfAnd10',
 'LILATracts_1And20',
 'LILATracts_Vehicle',
 'HUNVFlag',
 'LowIncomeTracts',
 'PovertyRate',
 'MedianFamilyIncome',
 'LA1and10',
 'LAhalfand10',
 'LA1and20',
 'LATracts_half',
 'LATracts1',
 'LATracts10',
 'LATracts20',
 'LATractsVehicle_20',
 'LAPOP1_10',
 'LAPOP05_10',
 'LAPOP1_20',
 'LALOWI1_10',
 'LALOWI05_10',
 'LALOWI1_20',
 'lapophalf',
 'lapophalfshare',
 'lalowihalf',
 'lalowihalfshare',
 'lakidshalf',
 'lakidshalfshare',
 'laseniorshalf',
 'laseniorshalfshare',
 'lawhitehalf',
 'lawhitehalfshare',
 'lablackhalf',
 'lablackhalfshare',
 'laasianhalf',
 'laasianhalfshare',
 'lanhopihalf',
 'lanhopihalfshare',
 'laaianhalf',
 'laaianhalfshare',
 'laomultirhalf',
 'laomultirhalfshare',
 'lahisphalf',
 'lahisphalfshare',
 'lahunvhalf',
 'lahunvhalfshare',
 'lasnaphalf',
 'lasnaphalfshare',
 'lapop1',
 'lapop1share',
 

In [9]:
fa_raw1 = pd.read_excel(FA_PATH, sheet_name=0)

print("Food Access columns (sample):", list(fa_raw1.columns)[:25])


Food Access columns (sample): ['Notes about the Food Access Research Atlas download data:']


In [10]:
fa_raw2 = pd.read_excel(FA_PATH, sheet_name=1)

print("Food Access columns (sample):", list(fa_raw2.columns)[:25])


Food Access columns (sample): ['Field', 'LongName', 'Description']


In [11]:
fa_raw3= pd.read_excel(FA_PATH, sheet_name=2)

print("Food Access columns (sample):", list(fa_raw3.columns)[:25])


Food Access columns (sample): ['CensusTract', 'State', 'County', 'Urban', 'Pop2010', 'OHU2010', 'GroupQuartersFlag', 'NUMGQTRS', 'PCTGQTRS', 'LILATracts_1And10', 'LILATracts_halfAnd10', 'LILATracts_1And20', 'LILATracts_Vehicle', 'HUNVFlag', 'LowIncomeTracts', 'PovertyRate', 'MedianFamilyIncome', 'LA1and10', 'LAhalfand10', 'LA1and20', 'LATracts_half', 'LATracts1', 'LATracts10', 'LATracts20', 'LATractsVehicle_20']


In [12]:

geo_cols = [c for c in food_access.columns if any(k in c.upper() for k in ["COUNTY", "TRACT", "CENSUS", "STATE"])]
geo_cols


['CensusTract',
 'State',
 'County',
 'LILATracts_1And10',
 'LILATracts_halfAnd10',
 'LILATracts_1And20',
 'LILATracts_Vehicle',
 'LowIncomeTracts',
 'LATracts_half',
 'LATracts1',
 'LATracts10',
 'LATracts20',
 'LATractsVehicle_20',
 'TractLOWI',
 'TractKids',
 'TractSeniors',
 'TractWhite',
 'TractBlack',
 'TractAsian',
 'TractNHOPI',
 'TractAIAN',
 'TractOMultir',
 'TractHispanic',
 'TractHUNV',
 'TractSNAP']

Dataset2



In [5]:

env_path = "FoodEnvironmentAtlas.xls"

xls_env = pd.ExcelFile(env_path)
xls_env.sheet_names


['Read_Me',
 ' Variable List',
 'Supplemental Data - County',
 'Supplemental Data - State',
 'ACCESS',
 'STORES',
 'RESTAURANTS',
 'ASSISTANCE',
 'INSECURITY',
 'TAXES',
 'LOCAL',
 'HEALTH',
 'SOCIOECONOMIC']

In [6]:
stores = pd.read_excel(
    env_path,
    sheet_name="STORES"
)

stores.head()

,FIPS,State,County,GROC11,GROC16,PCH_GROC_11_16,GROCPTH11,GROCPTH16,PCH_GROCPTH_11_16,SUPERC11,SUPERC16,PCH_SUPERC_11_16,SUPERCPTH11,SUPERCPTH16,PCH_SUPERCPTH_11_16,CONVS11,CONVS16,PCH_CONVS_11_16,CONVSPTH11,CONVSPTH16,PCH_CONVSPTH_11_16,SPECS11,SPECS16,PCH_SPECS_11_16,SPECSPTH11,SPECSPTH16,PCH_SPECSPTH_11_16,SNAPS12,SNAPS17,PCH_SNAPS_12_17,SNAPSPTH12,SNAPSPTH17,PCH_SNAPSPTH_12_17,WICS11,WICS16,PCH_WICS_11_16,WICSPTH11,WICSPTH16,PCH_WICSPTH_11_16
0,1001,AL,Autauga,5,3,-40.000000,0.090581,0.054271,-40.085748,1,1,0.000000,0.018116,0.018090,-0.142914,31,31,0.000000,0.561604,0.560802,-0.142914,1,1,0.000000,0.018116,0.018090,-0.142914,37.416667,44.666667,19.376392,0.674004,0.804747,19.397900,5.0,5.0,0.000000,0.090567,0.090511,-0.061543
1,1003,AL,Baldwin,27,29,7.407407,0.144746,0.139753,-3.449328,6,7,16.666667,0.032166,0.033733,4.874005,107,118,10.280374,0.573622,0.568650,-0.866761,20,27,35.000000,0.107219,0.130115,21.354206,138.333333,189.416667,36.927711,0.725055,0.890836,22.864524,26.0,28.0,7.692307,0.139380,0.134802,-3.284727
2,1005,AL,Barbour,6,4,-33.333333,0.219370,0.155195,-29.254287,0,1,NaN,0.000000,0.038799,NaN,22,19,-13.636364,0.804358,0.737177,-8.352145,3,2,-33.333333,0.109685,0.077598,-29.254287,34.833333,36.000000,3.349282,1.280590,1.424614,11.246689,7.0,6.0,-14.285714,0.255942,0.232387,-9.203081
3,1007,AL,Bibb,6,5,-16.666667,0.263794,0.220916,-16.254289,1,1,0.000000,0.043966,0.044183,0.494853,19,15,-21.052632,0.835348,0.662749,-20.661958,0,0,0.000000,0.000000,0.000000,0.000000,16.250000,18.166667,11.794872,0.719122,0.801423,11.444711,6.0,5.0,-16.666666,0.263771,0.221474,-16.035471
4,1009,AL,Blount,7,5,-28.571429,0.121608,0.086863,-28.571429,1,1,0.000000,0.017373,0.017373,0.000000,30,27,-10.000000,0.521177,0.469059,-10.000000,1,0,-100.000000,0.017373,0.000000,-100.000000,38.000000,40.166667,5.701754,0.657144,0.692374,5.361034,8.0,8.0,0.000000,0.139000,0.139089,0.064332


In [13]:
stores.columns.tolist()


['FIPS',
 'State',
 'County',
 'GROC11',
 'GROC16',
 'PCH_GROC_11_16',
 'GROCPTH11',
 'GROCPTH16',
 'PCH_GROCPTH_11_16',
 'SUPERC11',
 'SUPERC16',
 'PCH_SUPERC_11_16',
 'SUPERCPTH11',
 'SUPERCPTH16',
 'PCH_SUPERCPTH_11_16',
 'CONVS11',
 'CONVS16',
 'PCH_CONVS_11_16',
 'CONVSPTH11',
 'CONVSPTH16',
 'PCH_CONVSPTH_11_16',
 'SPECS11',
 'SPECS16',
 'PCH_SPECS_11_16',
 'SPECSPTH11',
 'SPECSPTH16',
 'PCH_SPECSPTH_11_16',
 'SNAPS12',
 'SNAPS17',
 'PCH_SNAPS_12_17',
 'SNAPSPTH12',
 'SNAPSPTH17',
 'PCH_SNAPSPTH_12_17',
 'WICS11',
 'WICS16',
 'PCH_WICS_11_16',
 'WICSPTH11',
 'WICSPTH16',
 'PCH_WICSPTH_11_16']

In [14]:
assistance = pd.read_excel(
    env_path,
    sheet_name="ASSISTANCE"
)

assistance.head()

,FIPS,State,County,REDEMP_SNAPS12,REDEMP_SNAPS17,PCH_REDEMP_SNAPS_12_17,PCT_SNAP12,PCT_SNAP17,PCH_SNAP_12_17,PC_SNAPBEN12,PC_SNAPBEN17,PCH_PC_SNAPBEN_12_17,SNAP_PART_RATE11,SNAP_PART_RATE16,SNAP_OAPP09,SNAP_OAPP16,SNAP_CAP09,SNAP_CAP16,SNAP_BBCE09,SNAP_BBCE16,SNAP_REPORTSIMPLE09,SNAP_REPORTSIMPLE16,PCT_NSLP12,PCT_NSLP17,PCH_NSLP_12_17,...,PCT_SFSP12,PCT_SFSP17,PCH_SFSP_12_17,PC_WIC_REDEMP11,PC_WIC_REDEMP16,PCH_PC_WIC_REDEMP_11_16,REDEMP_WICS11,REDEMP_WICS16,PCH_REDEMP_WICS_11_16,PCT_WIC12,PCT_WIC17,PCH_WIC_12_17,PCT_WICINFANTCHILD14,PCT_WICINFANTCHILD16,PCH_WICINFANTCHILD_14_16,PCT_WICWOMEN14,PCT_WICWOMEN16,PCH_WICWOMEN_14_16,PCT_CACFP12,PCT_CACFP17,PCH_CACFP_12_17,FDPIR12,FDPIR15,PCH_FDPIR_12_15,FOOD_BANKS18
0,1001,AL,Autauga,301432.081069,223185.678358,-25.958220,18.908476,16.500056,-2.40842,18.471487,16.354677,-11.459878,84.02,86.898,0.0,1.0,0,0,0,1,1,1,68.226043,63.12659,-5.099454,...,3.16032,6.369006,3.208686,15.612932,14.620244,-6.358115,172391.750000,161530.296875,-6.300448,2.947682,2.54357,-0.404113,33.481211,32.910876,-0.570336,3.318827,3.309759,-0.009068,0.891239,1.258763,0.367524,0,0,0.0,0
1,1003,AL,Baldwin,274394.503663,157623.133568,-42.556016,18.908476,16.500056,-2.40842,15.890722,11.479360,-27.760618,84.02,86.898,0.0,1.0,0,0,0,1,1,1,68.226043,63.12659,-5.099454,...,3.16032,6.369006,3.208686,17.107496,13.873837,-18.901999,122739.710938,102920.085938,-16.147688,2.947682,2.54357,-0.404113,33.481211,32.910876,-0.570336,3.318827,3.309759,-0.009068,0.891239,1.258763,0.367524,0,0,0.0,0
2,1005,AL,Barbour,325496.560766,257032.078889,-21.033857,18.908476,16.500056,-2.40842,31.116222,29.122147,-6.408476,84.02,86.898,0.0,1.0,0,0,0,1,1,1,68.226043,63.12659,-5.099454,...,3.16032,6.369006,3.208686,21.934052,24.032284,9.566095,85699.468750,103414.921875,20.671602,2.947682,2.54357,-0.404113,33.481211,32.910876,-0.570336,3.318827,3.309759,-0.009068,0.891239,1.258763,0.367524,0,0,0.0,0
3,1007,AL,Bibb,356444.032000,227725.245138,-36.111921,18.908476,16.500056,-2.40842,22.435049,17.557791,-21.739460,84.02,86.898,0.0,1.0,0,0,0,1,1,1,68.226043,63.12659,-5.099454,...,3.16032,6.369006,3.208686,21.482937,22.081812,2.787678,81445.398438,99703.796875,22.417961,2.947682,2.54357,-0.404113,33.481211,32.910876,-0.570336,3.318827,3.309759,-0.009068,0.891239,1.258763,0.367524,0,0,0.0,0
4,1009,AL,Blount,229730.022632,142860.825560,-37.813602,18.908476,16.500056,-2.40842,20.272305,12.377973,-38.941463,84.02,86.898,0.0,1.0,0,0,0,1,1,1,68.226043,63.12659,-5.099454,...,3.16032,6.369006,3.208686,17.110687,13.860495,-18.995102,123098.562500,99651.757812,-19.047180,2.947682,2.54357,-0.404113,33.481211,32.910876,-0.570336,3.318827,3.309759,-0.009068,0.891239,1.258763,0.367524,0,0,0.0,0


Dataset 3



In [ ]:
dp03_raw = pd.read_csv("ACSDP5Y2019.DP03-Data.csv", low_memory=False)


In [ ]:
dp03_raw.head()


,GEO_ID,NAME,DP03_0001E,DP03_0001M,DP03_0002E,DP03_0002M,DP03_0003E,DP03_0003M,DP03_0004E,DP03_0004M,DP03_0005E,DP03_0005M,DP03_0006E,DP03_0006M,DP03_0007E,DP03_0007M,DP03_0008E,DP03_0008M,DP03_0009E,DP03_0009M,DP03_0010E,DP03_0010M,DP03_0011E,DP03_0011M,DP03_0012E,...,DP03_0126PE,DP03_0126PM,DP03_0127PE,DP03_0127PM,DP03_0128PE,DP03_0128PM,DP03_0129PE,DP03_0129PM,DP03_0130PE,DP03_0130PM,DP03_0131PE,DP03_0131PM,DP03_0132PE,DP03_0132PM,DP03_0133PE,DP03_0133PM,DP03_0134PE,DP03_0134PM,DP03_0135PE,DP03_0135PM,DP03_0136PE,DP03_0136PM,DP03_0137PE,DP03_0137PM,Unnamed: 550
0,Geography,Geographic Area Name,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Civilian labor force,Margin of Error!!EMPLOYMENT STATUS!!Civilian l...,Estimate!!EMPLOYMENT STATUS!!Civilian labor fo...,Margin of Error!!EMPLOYMENT STATUS!!Civilian l...,Estimate!!EMPLOYMENT STATUS!!Females 16 years ...,Margin of Error!!EMPLOYMENT STATUS!!Females 16...,Estimate!!EMPLOYMENT STATUS!!Females 16 years ...,Margin of Error!!EMPLOYMENT STATUS!!Females 16...,Estimate!!EMPLOYMENT STATUS!!Females 16 years ...,...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,NaN
1,0500000US01001,"Autauga County, Alabama",43953,259,25908,671,25458,689,24522,738,936,246,450,185,18045,702,25458,689,(X),(X),22813,233,11750,516,11700,...,50.2,12.7,20.6,20.7,15.2,1.8,23.2,4.0,23.1,3.9,21.2,6.2,23.7,4.3,12.7,1.6,13.6,1.8,8.7,2.1,13.8,2.2,23.1,3.4,NaN
2,0500000US01003,"Baldwin County, Alabama",172297,329,99495,1720,99317,1737,95091,1811,4226,580,178,100,72802,1783,99317,1737,(X),(X),89385,245,47217,1287,47154,...,31.8,5.6,33.7,17.6,10.4,0.9,13.4,2.0,13.2,2.0,16.0,3.7,12.2,2.1,9.5,0.8,10.2,1.0,7.4,1.4,8.0,1.0,21.9,2.2,NaN
3,0500000US01005,"Barbour County, Alabama",20636,101,9262,444,9262,444,8413,448,849,176,0,22,11374,429,9262,444,(X),(X),9637,66,4304,252,4304,...,69.0,7.4,57.7,24.2,30.7,2.4,50.1,4.9,50.1,4.9,59.0,7.1,47.1,5.6,24.7,2.4,27.6,2.8,16.8,3.1,29.5,2.7,36.3,5.6,NaN
4,0500000US01007,"Bibb County, Alabama",18492,140,9046,567,9046,567,8387,615,659,248,0,22,9446,553,9046,567,(X),(X),8555,149,4243,361,4243,...,75.5,21.4,0.0,93.8,18.1,4.5,29.5,10.5,28.8,10.6,17.3,12.6,33.4,12.0,14.9,3.6,17.1,4.4,6.9,3

In [ ]:
dp04_raw = pd.read_csv("ACSDP5Y2019.DP04-Data.csv", low_memory=False)


In [ ]:
dp04_raw.head()

,GEO_ID,NAME,DP04_0001E,DP04_0001M,DP04_0002E,DP04_0002M,DP04_0003E,DP04_0003M,DP04_0004E,DP04_0004M,DP04_0005E,DP04_0005M,DP04_0006E,DP04_0006M,DP04_0007E,DP04_0007M,DP04_0008E,DP04_0008M,DP04_0009E,DP04_0009M,DP04_0010E,DP04_0010M,DP04_0011E,DP04_0011M,DP04_0012E,...,DP04_0132PE,DP04_0132PM,DP04_0133PE,DP04_0133PM,DP04_0134PE,DP04_0134PM,DP04_0135PE,DP04_0135PM,DP04_0136PE,DP04_0136PM,DP04_0137PE,DP04_0137PM,DP04_0138PE,DP04_0138PM,DP04_0139PE,DP04_0139PM,DP04_0140PE,DP04_0140PM,DP04_0141PE,DP04_0141PM,DP04_0142PE,DP04_0142PM,DP04_0143PE,DP04_0143PM,Unnamed: 574
0,Geography,Geographic Area Name,Estimate!!HOUSING OCCUPANCY!!Total housing units,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,Estimate!!HOUSING OCCUPANCY!!Total housing uni...,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,Estimate!!HOUSING OCCUPANCY!!Total housing uni...,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,Estimate!!HOUSING OCCUPANCY!!Total housing uni...,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,Estimate!!HOUSING OCCUPANCY!!Total housing uni...,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,Estimate!!UNITS IN STRUCTURE!!Total housing units,Margin of Error!!UNITS IN STRUCTURE!!Total hou...,Estimate!!UNITS IN STRUCTURE!!Total housing un...,Margin of Error!!UNITS IN STRUCTURE!!Total hou...,Estimate!!UNITS IN STRUCTURE!!Total housing un...,Margin of Error!!UNITS IN STRUCTURE!!Total hou...,Estimate!!UNITS IN STRUCTURE!!Total housing un...,Margin of Error!!UNITS IN STRUCTURE!!Total hou...,Estimate!!UNITS IN STRUCTURE!!Total housing un...,Margin of Error!!UNITS IN STRUCTURE!!Total hou...,Estimate!!UNITS IN STRUCTURE!!Total housing un...,Margin of Error!!UNITS IN STRUCTURE!!Total hou...,Estimate!!UNITS IN STRUCTURE!!Total housing un...,...,Percent!!GROSS RENT!!Occupied units paying ren...,Percent Margin of Error!!GROSS RENT!!Occupied ...,Percent!!GROSS RENT!!Occupied units paying ren...,Percent Margin of Error!!GROSS RENT!!Occupied ...,Percent!!GROSS RENT!!Occupied units paying ren...,Percent Margin of Error!!GROSS RENT!!Occupied ...,Percent!!GROSS RENT!!Occupied units paying ren...,Percent Margin of Error!!GROSS RENT!!Occupied ...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,NaN
1,0500000US01001,"Autauga County, Alabama",23493,83,21397,325,2096,313,1.4,0.6,3.1,1.7,23493,83,16748,463,216,118,118,57,457,147,670,237,390,...,0.0,0.6,3.2,2.2,(X),(X),(X),(X),4905,(X),12.5,3.7,10.2,3.4,17.3,4.9,11.4,4.9,8.8,3.5,39.9,6.6,(X),(X),NaN
2,0500000US01003,"Baldwin County, Alabama",114164,219,80930,1127,33234,1128,3.3,0.7,43.6,2.2,114164,219,72583,1155,1290,247,2161,388,2448,409,3135,434,3757,...,0.4,0.4,1.2,0.9,(X),(X),(X),(X),17335,(X),12.7,2.6,14.3,2.4,13.8,2.2,12.2,2.1,9.6,2.0,37.4,3.5,(X),(X),NaN
3,0500000US01005,"Barbour County, Alabama",12013,143,9345,313,2668,267,3.8,1.4,7.4,3.0,12013,143,6573,325,124,59,745,158,453,113,376,109,115,...,0.0,1.0,0.0,1.0,(X),(X),(X),(X),2979,(X),15.1,4.2,11.7,4.0,14.3,4.1,9.6,3.3,8.4,3.4,40.9,5.4,(X),(X),NaN
4,0500000US01007,"Bibb County, Alabama",9185,68,6891,333,2294,328,1.5,1.3,5.6,5.4,9185,68,5532,353,33,33,0,22,194,141,268,135,71,...,0.0,2.2,0.0,2.2,(X),(X),(X),(X),1406,(X),25.5,9.0,6.7,5.9,7.7,4.8,11.6,6.3,8.0,5.0,40.5,10.8,(X),(X),NaN


In [ ]:
dp05_raw = pd.read_csv("ACSDP5Y2019.DP05-Data.csv", low_memory=False)


In [ ]:
dp05_raw.head()

,GEO_ID,NAME,DP05_0001E,DP05_0001M,DP05_0002E,DP05_0002M,DP05_0003E,DP05_0003M,DP05_0004E,DP05_0004M,DP05_0005E,DP05_0005M,DP05_0006E,DP05_0006M,DP05_0007E,DP05_0007M,DP05_0008E,DP05_0008M,DP05_0009E,DP05_0009M,DP05_0010E,DP05_0010M,DP05_0011E,DP05_0011M,DP05_0012E,...,DP05_0078PE,DP05_0078PM,DP05_0079PE,DP05_0079PM,DP05_0080PE,DP05_0080PM,DP05_0081PE,DP05_0081PM,DP05_0082PE,DP05_0082PM,DP05_0083PE,DP05_0083PM,DP05_0084PE,DP05_0084PM,DP05_0085PE,DP05_0085PM,DP05_0086PE,DP05_0086PM,DP05_0087PE,DP05_0087PM,DP05_0088PE,DP05_0088PM,DP05_0089PE,DP05_0089PM,Unnamed: 358
0,Geography,Geographic Area Name,Estimate!!SEX AND AGE!!Total population,Margin of Error!!SEX AND AGE!!Total population,Estimate!!SEX AND AGE!!Total population!!Male,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!Female,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!Sex r...,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!Under...,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!5 to ...,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!10 to...,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!15 to...,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!20 to...,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!25 to...,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!35 to...,Margin of Error!!SEX AND AGE!!Total population...,Estimate!!SEX AND AGE!!Total population!!45 to...,...,Percent!!HISPANIC OR LATINO AND RACE!!Total po...,Percent Margin of Error!!HISPANIC OR LATINO AN...,Percent!!HISPANIC OR LATINO AND RACE!!Total po...,Percent Margin of Error!!HISPANIC OR LATINO AN...,Percent!!HISPANIC OR LATINO AND RACE!!Total po...,Percent Margin of Error!!HISPANIC OR LATINO AN...,Percent!!HISPANIC OR LATINO AND RACE!!Total po...,Percent Margin of Error!!HISPANIC OR LATINO AN...,Percent!!HISPANIC OR LATINO AND RACE!!Total po...,Percent Margin of Error!!HISPANIC OR LATINO AN...,Percent!!HISPANIC OR LATINO AND RACE!!Total po...,Percent Margin of Error!!HISPANIC OR LATINO AN...,Percent!!HISPANIC OR LATINO AND RACE!!Total po...,Percent Margin of Error!!HISPANIC OR LATINO AN...,Percent!!HISPANIC OR LATINO AND RACE!!Total po...,Percent Margin of Error!!HISPANIC OR LATINO AN...,Percent!!Total housing units,Percent Margin of Error!!Total housing units,"Percent!!CITIZEN, VOTING AGE POPULATION!!Citiz...","Percent Margin of Error!!CITIZEN, VOTING AGE P...","Percent!!CITIZEN, VOTING AGE POPULATION!!Citiz...","Percent Margin of Error!!CITIZEN, VOTING AGE P...","Percent!!CITIZEN, VOTING AGE POPULATION!!Citiz...","Percent Margin of Error!!CITIZEN, VOTING AGE P...",NaN
1,0500000US01001,"Autauga County, Alabama",55380,*****,26934,166,28446,166,94.7,1.1,3217,107,3814,352,3600,350,3812,211,3570,220,7056,202,7306,182,7736,...,19.0,0.5,0.3,0.1,1.0,0.3,0.0,0.1,0.2,0.2,2.1,0.5,0.0,0.1,2.1,0.5,(X),(X),41647,(X),47.6,0.2,52.4,0.2,NaN
2,0500000US01003,"Baldwin County, Alabama",212830,*****,103496,233,109334,233,94.7,0.4,11689,30,12058,831,14262,859,12831,285,10878,309,23942,323,25834,293,28412,...,9.2,0.2,0.7,0.1,0.9,0.1,0.0,0.1,0.2,0.1,1.5,0.3,0.0,0.1,1.4,0.3,(X),(X),162883,(X),47.9,0.2,52.1,0.2,NaN
3,0500000US01005,"Barbour County, Alabama",25361,*****,13421,80,11940,80,112.4,1.4,1349,26,1622,171,1422,170,1412,44,1592,52,3642,87,3038,68,3310,...,47.4,0.6,0.3,0.2,0.5,0.1,0.0,0.1,0.4,0.3,1.2,0.5,0.1,0.1,1.1,0.5,(X),(X),19728,(X),53.4,0.3,46.6,0.3,NaN
4,0500000US01007,"Bibb County, Alabama",22493,*****,12150,193,10343,193,117.5,4.1,1315,170,1219,270,1132,263,1400,129,1259,196,3276,304,2910,228,3519,...,22.1,0.5,0.1,0.3,0.1,0.2,0.0,0.1,0.0,0.1,0.5,0.3,0.0,0.1,0.5,0.3,(X),(X),17662,(X),53.6,0.4,46.4,0.4,NaN
